In [2]:

#!/usr/bin/env python3
"""
Probe EarthScope for the 1991 Pinatubo XB dataset.

Install:
    pip install obspy requests

Run:
    python probe_pinatubo_earthscope.py

Outputs:
    pinatubo_earthscope_probe/
        station_metadata.xml
        station_summary.txt
        waveform_*.mseed
"""

from pathlib import Path
from datetime import timedelta
import requests
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNException

NETWORK = "XB"
OUTDIR = Path("pinatubo_earthscope_probe")
OUTDIR.mkdir(exist_ok=True)

# EarthScope's FDSN service endpoint.
BASE_URL = "https://service.earthscope.org"

# Historical network epoch and eruption interval.
START = UTCDateTime("1991-01-01")
END = UTCDateTime("1992-01-01")

# A few strategically chosen test dates.
TEST_DATES = [
    "1991-03-15",
    "1991-04-05",
    "1991-06-12",
    "1991-06-15",
    "1991-07-01",
    "1991-09-01",
]

def make_client():
    for url in [BASE_URL, "IRIS"]:
        try:
            client = Client(url, timeout=30)
            print(f"Connected to {url}")
            return client
        except Exception as exc:
            print(f"Client failed ({url}): {exc}")
    return None


def check_http_services():
    print("\n=== HTTP SERVICE CHECK ===")

    urls = [
        f"{BASE_URL}/fdsnws/station/1/version",
        f"{BASE_URL}/fdsnws/dataselect/1/version",
        f"{BASE_URL}/fdsnws/availability/1/version",
    ]

    for url in urls:
        try:
            response = requests.get(url, timeout=20)
            print(f"{response.status_code}: {url}")
            print(response.text[:200])
        except requests.RequestException as exc:
            print(f"ERROR: {url}: {exc}")


def get_inventory(client):
    print("\n=== STATION METADATA ===")

    for level in ("channel", "station", "network"):
        try:
            inv = client.get_stations(
                network=NETWORK,
                starttime=START,
                endtime=END,
                level=level,
            )

            print(f"Success at level={level}")
            print(inv)

            inv.write(
                str(OUTDIR / "station_metadata.xml"),
                format="STATIONXML",
            )

            with open(OUTDIR / "station_summary.txt", "w") as f:
                f.write(str(inv))

            return inv

        except Exception as exc:
            print(f"Failed at level={level}: {exc}")

    return None


def list_channels(inv):
    channels = []

    if inv is None:
        return channels

    for network in inv:
        for station in network:
            for channel in station:
                channels.append(
                    (
                        network.code,
                        station.code,
                        channel.location_code or "",
                        channel.code,
                    )
                )

    print(f"\nFound {len(channels)} channel epochs")

    for item in channels[:30]:
        print(".".join(item))

    return list(dict.fromkeys(channels))


def probe_waveforms(client, channels):
    print("\n=== WAVEFORM TESTS ===")

    # If metadata has no channel information, try wildcard
    # requests. These can work even without station metadata.
    if not channels:
        channels = [(NETWORK, "*", "*", "*")]

    # Limit requests to avoid excessive server load.
    channels = channels[:5]

    successes = []

    for date in TEST_DATES:
        t0 = UTCDateTime(date)
        t1 = t0 + 86400

        for net, sta, loc, cha in channels:
            print(
                f"Trying {net}.{sta}.{loc}.{cha} "
                f"{t0} to {t1}"
            )

            try:
                st = client.get_waveforms(
                    network=net,
                    station=sta,
                    location=loc,
                    channel=cha,
                    starttime=t0,
                    endtime=t1,
                )

                if not st:
                    continue

                print("SUCCESS!")
                print(st)

                safe_name = (
                    f"{net}_{sta}_{loc or '--'}_{cha}_"
                    f"{t0.strftime('%Y%m%d')}.mseed"
                )

                outfile = OUTDIR / safe_name
                st.write(str(outfile), format="MSEED")
                print(f"Saved: {outfile}")

                successes.append(str(outfile))

            except FDSNException as exc:
                print(f"FDSN response: {exc}")

            except Exception as exc:
                print(f"ERROR: {exc}")

    return successes


def check_availability():
    print("\n=== AVAILABILITY SERVICE ===")

    url = (
        f"{BASE_URL}/fdsnws/availability/1/query"
    )

    params = {
        "net": NETWORK,
        "sta": "*",
        "loc": "*",
        "cha": "*",
        "starttime": str(START),
        "endtime": str(END),
        "format": "text",
    }

    try:
        response = requests.get(
            url, params=params, timeout=60
        )

        print("HTTP status:", response.status_code)
        print(response.text[:5000])

        if response.ok:
            (OUTDIR / "availability.txt").write_text(
                response.text
            )

    except requests.RequestException as exc:
        print("Availability query failed:", exc)


def main():
    check_http_services()

    client = make_client()

    if client is None:
        print("Could not initialize FDSN client.")
        return

    inv = get_inventory(client)
    channels = list_channels(inv)

    check_availability()

    successes = probe_waveforms(client, channels)

    print("\n=== SUMMARY ===")
    print("Network:", NETWORK)
    print("Channel epochs found:", len(channels))
    print("Waveform files downloaded:", len(successes))
    print("Output directory:", OUTDIR.resolve())

    if not successes:
        print(
            "\nNo waveforms retrieved. This does not "
            "necessarily mean the data are absent. "
            "They may be held in a nonstandard archive "
            "or have unusable timing metadata."
        )


if __name__ == "__main__":
    main()



=== HTTP SERVICE CHECK ===
200: https://service.earthscope.org/fdsnws/station/1/version
1.1.58
200: https://service.earthscope.org/fdsnws/dataselect/1/version
1.1.82
404: https://service.earthscope.org/fdsnws/availability/1/version
This service has been retired
Connected to https://service.earthscope.org

=== STATION METADATA ===
Failed at level=channel: No data available for request.
HTTP Status code: 204
Detailed response of server:


Failed at level=station: No data available for request.
HTTP Status code: 204
Detailed response of server:


Failed at level=network: No data available for request.
HTTP Status code: 204
Detailed response of server:



=== AVAILABILITY SERVICE ===
HTTP status: 404
This service has been retired

=== WAVEFORM TESTS ===
Trying XB.*.*.* 1991-03-15T00:00:00.000000Z to 1991-03-16T00:00:00.000000Z
FDSN response: No data available for request.
HTTP Status code: 204
Detailed response of server:


Trying XB.*.*.* 1991-04-05T00:00:00.000000Z to 1991-04-06T00:00:0